In [82]:
import os
import json
import pandas as pd
from maomao.utils.constants import *
from maomao.hierarchical_structure.define_hierarchy_and_structure import (
    generate_sequence_id,
    normalize_sequence,
)

#### Characterization of negative sequences by category of evidence
- This notebook analyzes the provenance and biological meaning of negative labels in the peptide dataset by linking each sequence to the type of experimental or curated evidence supporting its negative annotation.

- As input, it loads the curated dataset restricted to sequences classified as negative after label-consistency analysis, together with an external evidence table that categorizes negative datasets according to their annotation criteria (e.g., experimentally validated non-hemolytic peptides, inferred negatives, or decoy datasets). Only source-level label columns are retained to ensure a clean mapping between sequences and their originating datasets.

- For each negative sequence, the notebook identifies the contributing data sources and expands sequence–source relationships into individual records. These are then merged with the evidence metadata to associate each sequence with one or more negative evidence categories. The resulting information is aggregated into a sequence-by-evidence-category pivot table, explicitly encoding missing associations.

- Finally, summary statistics describing the distribution of negative evidence categories are computed and appended to the dataset metadata. Both the per-sequence evidence matrix and the updated metadata file are exported, providing transparency into the composition of the negative class and enabling downstream analyses of label reliability, dataset bias, and negative sampling strategies.

In [83]:
toxic_effect = "toxic" # Change for different toxic effects (e.g., toxic, neurotoxic, hemolytic, etc.)
integration_folder = f"../../processed_data/integrating_and_cleaning_data"

- Read data

In [84]:
df_evidence = (
    pd.read_excel("../../raw_data/evidence_negative_dataset.xlsx")
    .assign(task=lambda x: x["task"].str.lower())
    .loc[lambda x: x["task"].str.contains(f"{toxic_effect}", case=False, na=False)] # Filter data sources by activity

)
df_evidence

,name source,task,obtaining negative dataset,negative dataset category
0,BIOPEP-UWM,"celiac toxic, cytotoxic, hemolytic, toxic, emb...",No information,no negative data
1,CICERON,"celiac toxic, cytotoxic, hemolytic, embryotoxi...",No information,no negative data
2,Plantpepdb,"celiac toxic, cytotoxic, hemolytic, toxic, tox...",No information,no negative data
3,Peptipedia2.0,"cytotoxic, hemolytic, neurotoxic, toxic",No information,no negative data
4,MultiPep,"celiac toxic, hemolytic, embryotoxic, toxic, i...",No information,no negative data
18,Toropov et al.,cytotoxic,No information,no information
19,DRAMP,"cytotoxic, cytolytic, toxic, insecticidal, hem...",Unrelated activities,weak or unconfirmed negatives
20,iAMPCN,"cytotoxic, hemolytic, insecticidal, toxic, ant...",Sampling from uniprot,weak or unconfirmed negatives
23,AMPDB,"cytotoxic, hemolytic, platelet aggregation inh...",No information,no negative data
24,embryoTox,embryotoxic,No information,no information


In [85]:
df_only_negative = (
    pd.read_csv(f"{integration_folder}/{toxic_effect}/negative.csv")
      .loc[:, lambda df: ~df.columns.str.contains("unlabel")]
      .iloc[:, :-7]
)

In [86]:
df_only_negative

,sequence,BIOPEP-UWM,CAPTP,CICERON,CSM-Toxin,HyPepTox-Fuse,iAMPCN,MultiPep,Pep-Lab_db,peptidereactor,...,ToxinPred 3.0,ToxiPep,ToxMSRC,ToxTeller,TPpred-LE,UniDL4BioPep,Zhao et al.,Peptipedia2.0,counts_1,counts_0
0,MSLLPVMVIFGLSFPPVFFELLVPLALFFLLRRLLQPTGIYDFVWH...,999,999,999,999,999,999,999,999,999,...,999,999,999,999,999,999,999,999,0,1
1,INWKKWWQVFYTVV,999,999,999,999,999,0,999,999,999,...,999,999,999,999,999,999,999,999,0,1
2,MIPVRCLSCGKPVSAYFNEYQRRVADGEDPKDVLDDLGLKRYCCRR...,999,999,999,0,999,999,999,999,999,...,999,999,999,999,999,999,999,999,0,2
3,KRCHLTIDKATACSLSDCRLSCYSGYNGVGKCFDDPKVAGPSNCGC...,999,999,999,999,999,0,999,999,999,...,999,999,999,999,999,999,999,999,0,1
4,MYPIQIVFSENPIDQRHLGQSGGTISFTACGLPVFHFETQEQFQAY...,999,999,999,0,999,999,999,999,999,...,999,999,999,999,999,999,999,999,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37514,RICRTRLTRRAGNSL,999,999,999,999,999,0,999,999,999,...,999,999,999,999,999,999,999,999,0,1
37515,MVFGTVKEVSVNFIESIFKTFKIVEEISNTNLINCTFNNNNPIQQS...,999,999,999,0,999,999,999,999,999,...,999,999,999,999,999,999,999,999,0,2
37516,LVGLVFPAIAMASLFLYVQKNKIV,999,999,999,999,999,999,999,999,999,...,999,999,999,999,999,999,999,999,0,1
37517,MHSTPLVSAALIFAYTLTSDRTPAHASCTHTMMRTVRHTFLHTNST...,999,999,999,0,999,999,999,999,999,...,999,999,999,999,999,999,999,999,0,2


- Create dataset pivote

In [87]:
# Function to create the 'name source' column with the column names that have values ​​other than 999
def create_name_source(row):
    return [col for col in row.index[1:] if row[col] != 999]

# Exclude the 'sequence' column

# Apply the function row by row
df_only_negative['name source'] = df_only_negative.apply(create_name_source, axis=1)

# Expand the 'name source' lists into individual rows
df_exploded = df_only_negative.explode('name source').reset_index(drop=True)

# Final result
df_result = df_exploded[['sequence', 'name source']]

In [88]:
df_exploded = df_exploded.merge(df_evidence[['name source', 'negative dataset category']], on='name source', how='left')
df_exploded

,sequence,BIOPEP-UWM,CAPTP,CICERON,CSM-Toxin,HyPepTox-Fuse,iAMPCN,MultiPep,Pep-Lab_db,peptidereactor,...,ToxMSRC,ToxTeller,TPpred-LE,UniDL4BioPep,Zhao et al.,Peptipedia2.0,counts_1,counts_0,name source,negative dataset category
0,MSLLPVMVIFGLSFPPVFFELLVPLALFFLLRRLLQPTGIYDFVWH...,999,999,999,999,999,999,999,999,999,...,999,999,999,999,999,999,0,1,ProToxin,weak or unconfirmed negatives
1,MSLLPVMVIFGLSFPPVFFELLVPLALFFLLRRLLQPTGIYDFVWH...,999,999,999,999,999,999,999,999,999,...,999,999,999,999,999,999,0,1,counts_1,NaN
2,MSLLPVMVIFGLSFPPVFFELLVPLALFFLLRRLLQPTGIYDFVWH...,999,999,999,999,999,999,999,999,999,...,999,999,999,999,999,999,0,1,counts_0,NaN
3,INWKKWWQVFYTVV,999,999,999,999,999,0,999,999,999,...,999,999,999,999,999,999,0,1,iAMPCN,weak or unconfirmed negatives
4,INWKKWWQVFYTVV,999,999,999,999,999,0,999,999,999,...,999,999,999,999,999,999,0,1,counts_1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126258,MHSTPLVSAALIFAYTLTSDRTPAHASCTHTMMRTVRHTFLHTNST...,999,999,999,0,999,999,999,999,999,...,999,999,999,999,999,999,0,2,counts_0,NaN
126259,GFGCPFNLNECHAHCLSIGRKFGFCAGPLRATCTCGKQ,999,999,999,999,999,0,999,999,999,...,999,999,999,999,999,999,0,2,iAMPCN,weak or unconfirmed negatives
126260,GFGCPFNLNECHAHCLSIGRKFGFCAGPLRATCTCGKQ,999,999,999,999,999,0,999,999,999,...,999,999,999,999,999,999,0,2,tAMPer,weak or unconfirmed negatives
126261,GFGCPFNLNECHAHCLSIGRKFGFCAGPLRATCTCGKQ,999,999,999,999,999,0,999,999,999,...,999,999,999,999,999,999,0,2,counts_1,NaN


In [89]:
df_exploded["negative dataset category"].value_counts()

negative dataset category
weak or unconfirmed negatives    50682
strong negatives                   543
Name: count, dtype: int64

In [90]:
df_unique_sequences = df_exploded[['sequence', 'negative dataset category']].drop_duplicates()

# Pivotar el DataFrame para que las secuencias sean filas y los toxicity target las columnas
df_pivote_evidence = df_unique_sequences.pivot_table(index='sequence', columns='negative dataset category', aggfunc='size', fill_value=999)

In [91]:
df_pivote_evidence

negative dataset category,strong negatives,weak or unconfirmed negatives
sequence,,
AAAAAAAIKMLMDLVNERIMALNKKAKK,999,1
AAAAARRRIRKQAHAHSK,999,1
AAAADTIVLPYNDIDA,999,1
AAAAGDSAASDLLGDNILRSEDPPMSIDLTFHMLRNMIHMAKMEGEREQAQINRNLLDEV,999,1
AAAAGFEKGIDRDFEPVLFMTPLN,999,1
...,...,...
YYQANGGFLIAYQPL,999,1
YYQSYWTSMVS,999,1
YYQVSEERRRDLASLARLYALAR,999,1


- Working with metada

In [92]:
with open(f"{integration_folder}/{toxic_effect}/metadata.json", "r") as f:
    metadata = json.load(f)

In [93]:
evidence_counts = (
    df_pivote_evidence
    .replace(999, 0)
    .sum()
    .astype(int)
    .to_dict()
)

In [94]:
metadata["evidence_negative_dataset_statistics"] = {"category": evidence_counts}

- Add ID column

In [95]:
if "sequence" not in df_pivote_evidence.columns:
    df_pivote_evidence = (
        df_pivote_evidence.reset_index()
    )

df_pivote_evidence = df_pivote_evidence.drop(
    columns=["id"],
    errors="ignore",
)

normalized_sequences = df_pivote_evidence[
    "sequence"
].map(normalize_sequence)

if normalized_sequences.isna().any():
    raise ValueError(
        "Some evidence sequences could not be normalized."
    )

df_pivote_evidence["sequence"] = (
    normalized_sequences
)

if df_pivote_evidence["sequence"].duplicated().any():
    raise ValueError(
        "Duplicate sequences found after normalization."
    )

df_pivote_evidence.insert(
    0,
    "id",
    normalized_sequences.map(
        generate_sequence_id
    ),
)

if not df_pivote_evidence["id"].is_unique:
    raise ValueError(
        "Duplicate SHA-256 identifiers found."
    )

df_pivote_evidence.columns.name = None

df_pivote_evidence.head()

,id,sequence,strong negatives,weak or unconfirmed negatives
0,sha256_1de8b390ef5a86661bf0da4babf5a6059743fb6...,AAAAAAAIKMLMDLVNERIMALNKKAKK,999,1
1,sha256_42c3b829be38387546e22839e4d70387e2af55c...,AAAAARRRIRKQAHAHSK,999,1
2,sha256_d28a4510f4c0205ff4f9d7876d72ccbd1a53923...,AAAADTIVLPYNDIDA,999,1
3,sha256_8f0f6b0173fac930e041f1f05ce1a137b1ebe1a...,AAAAGDSAASDLLGDNILRSEDPPMSIDLTFHMLRNMIHMAKMEGE...,999,1
4,sha256_971d4d9249953f89c0d94e3df0c5989c61b8b91...,AAAAGFEKGIDRDFEPVLFMTPLN,999,1


- Exporting data

In [96]:
with open(f"{integration_folder}/{toxic_effect}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

In [97]:
df_pivote_evidence.to_csv(
    (
        f"{integration_folder}/{toxic_effect}/"
        "sequence_negative_evidece.csv"
    ),
    index=False,
)